# Equity models: calibrate and price

Result-driven: calibrate the equity models on one ticker, then price a put and a digital with each of them, with the
greeks to the market inputs, against Black-Scholes. How the models work: `local_vol.ipynb`, `heston.ipynb`,
`heston_slv.ipynb`.

In [ ]:
TICKER = "TICKER1"
STRIKE = 0.80            # strike / spot
EXPIRY_DAYS = 365
MC = dict(n_paths=50_000, n_steps=100, seed=1)

In [ ]:
import os
import time
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorquant as tq

mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")       # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity(TICKER)
EXPIRY = mkt.evaluation_date + timedelta(days=EXPIRY_DAYS)
K = round(STRIKE * float(eq.spot), 2)
print(f"{TICKER}  {mkt.evaluation_date}  spot {float(eq.spot):.2f}  K {K:.2f}  F(T) {float(eq.forward(EXPIRY)):.2f}")

## 1. Calibration

`Model.calibrate(equity)` fits once; `model.process(equity)` binds it to today's market (local vol grid, forwards).

In [ ]:
MODELS = {"local vol": tq.LocalVolatilityModel, "Heston": tq.HestonModel,
          "displaced diffusion": tq.DisplacedDiffusionModel, "Heston SLV": tq.HestonSLVModel}
models, timing = {}, []
for name, cls in MODELS.items():
    t0 = time.perf_counter()
    models[name] = cls.calibrate(eq)
    t1 = time.perf_counter()
    models[name].process(eq)
    timing.append({"model": name, "calibrate [s]": t1 - t0, "process [s]": time.perf_counter() - t1,
                   "parameters": {k: np.round(v.numpy(), 4).tolist() for k, v in models[name].parameters.items()}})
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(timing).set_index("model").round(3)

Fit per maturity (model - market implied vol, vol points). Local vol reprices the surface by construction; SLV holds the Heston fit and the leverage closes the gap.

In [ ]:
models["Heston"].report.set_index("maturity").round(2)

In [ ]:
models["displaced diffusion"].report.set_index("maturity").round(3)

## 2. Put 80%, 1Y

A trade is a `Product`; a pricer prices it under a `GradientTape`, and one gradient to `equity.risk_factors()` gives
all the greeks. For a calibrated model the greeks go through the calibration: `par_greeks` adds the model greeks
mapped by the stored Jacobian. Delta per unit of spot, vega per vol point (sum over the surface nodes), rho per bp
(sum over the curve pillars).

In [ ]:
SPOT, VOL = (next(k for k in eq.risk_factors() if k.startswith("EQ:") and k.endswith(s)) for s in (":SPOT", ":VOL"))


def summary(name, price, tape, model=None, se=0.0, seconds=None):
    params = model.parameters if model is not None else {}
    g = tape.gradient(price, {"model": params, "market": eq.risk_factors()})
    market = {k: [None if x is None else x.numpy() for x in v] for k, v in g["market"].items()}
    greeks = tq.par_greeks(model, {k: v.numpy() for k, v in g["model"].items()}, market) if model else \
        {k: [np.zeros(()) if x is None else x for x in v] for k, v in market.items()}
    rho = sum(np.sum(x) for k, v in greeks.items() if k.startswith("IR:") for x in v)
    return {"model": name, "price": float(price), "s.e.": float(se or 0.0),
            "delta": float(np.sum(greeks[SPOT][0])), "vega (1 vol pt)": float(np.sum(greeks[VOL][0])) / 100,
            "rho (1 bp)": float(rho) * 1e-4, "time [s]": seconds}, greeks


def black_scholes(product, name="Black-Scholes", **kwargs):
    pricer, t0 = tq.BlackScholesPricer(**kwargs), time.perf_counter()
    pricer.price(product, mkt, autodiff=True)
    return summary(name, product.price, pricer.tape, seconds=time.perf_counter() - t0)


def price_all(make_product, benchmarks):
    rows, vegas = [], {}
    for row, greeks in benchmarks:
        rows.append(row); vegas[row["model"]] = greeks[VOL][0]
    for name, model in models.items():
        product, pricer = make_product(), tq.VanillaMCPricer(model=model, **MC)
        t0 = time.perf_counter()
        pricer.price(product, mkt, autodiff=True)
        row, greeks = summary(name, product.price, pricer.tape, model, product.standard_error,
                              time.perf_counter() - t0)
        rows.append(row); vegas[name] = greeks[VOL][0]
    df = pd.DataFrame(rows).set_index("model")
    reference = rows[0]["model"]
    df.insert(2, f"vs {reference}", df["price"] - df.loc[reference, "price"])
    return df, vegas

In [ ]:
def put(strike=K):
    return tq.VanillaOption(eq.currency, start_date=mkt.evaluation_date, end_date=EXPIRY,
                            option_type=tq.OptionType.Put, strike=strike, underlying=TICKER)


put_table, put_vegas = price_all(put, [black_scholes(put())])
put_table.round(4)

Vega per maturity (sum over strikes, per vol point): Black-Scholes and the local vol family sit on the 1Y node, Heston spreads it over the surface through its calibration.

In [ ]:
def vega_by_maturity(vegas):
    return pd.DataFrame({m: np.sum(v, axis=1) / 100 for m, v in vegas.items()},
                        index=pd.Index(np.round(eq.vol_surface.maturity, 2), name="maturity"))


vega_by_maturity(put_vegas).round(4)

## 3. Digital put 80%, 1Y (cash-or-nothing, pays 100)

Benchmark: the **put spread** on the smile, `BlackScholesPricer(digital_spread=h)`:
`100 * (P(K + hK) - P(K - hK)) / 2hK` with each put at its own vol (a digital put is `dP/dK`), the market price of
the digital, no model. Black-Scholes at the vol of the strike (the default) misses the skew term,
`digital = BS digital + vega * dsigma/dK`, shown for reference. Monte Carlo smooths the indicator (sigmoid, 1% of the
strike) so the greeks exist.

In [ ]:
def digital_put():
    return tq.DigitalOption(eq.currency, mkt.evaluation_date, EXPIRY, tq.OptionType.Put, K,
                            payoff_type=tq.PayoffType.CashOrNothing, cash_amount=100.0, underlying=TICKER)


benchmarks = [black_scholes(digital_put(), "put spread (smile)", digital_spread=0.005), black_scholes(digital_put())]
digital_table, digital_vegas = price_all(digital_put, benchmarks)
digital_table.round(4)

In [ ]:
vega_by_maturity(digital_vegas).round(4)